In [ ]:
import collections
import matplotlib.pyplot as plt
from IPython import display
import itertools as itr
import numpy as np
from sklearn.metrics import mean_squared_error
from random import randrange
from tabulate import tabulate

import torch
from torch import nn
from torch import optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.func import functional_call, vmap, vjp, jvp, grad

from scipy.linalg import svdvals, norm
from scipy.special import softmax
from scipy.optimize import minimize_scalar

from common.const import DATASET_PATH
from common.util import *
from common.ffn_model.minst.ffn_minst_util import MetaData
from common.ffn_model.minst.ffn_minst_optimise import \
    StepCalculatorZhangLambda, StepCalculatorEtaSoftmaxArmihoNorm2, StepCalculatorEtaSoftmaxArmihoNorm1\
    , StepCalculatorEtaSoftmaxArmihoNorm2Base, StepCalculatorEtaSoftmaxArmihoNorm1Base\
        , StepCalculatorZhangSimplified, reduce_to_active
from common.ffn_model.minst.ffn_minst_relu import MNISTReLU

from torchvision.datasets import EMNIST
#from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid

import logging

In [ ]:
image_transform = ToTensor()

train_dataset = EMNIST(root=DATASET_PATH, split='letters', train=True, download=True, transform=image_transform)
test_dataset = EMNIST(root=DATASET_PATH, split='letters', train=False, download=True, transform=image_transform)

BATCH_SIZE = 128 #128 #64 #

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
)

In [ ]:
#Weights distribution variances are set as in (5.67)
slope_plus, slope_minus=1.0, 0.1
cb, cw = 0, 2.0/(slope_plus**2.0 + slope_minus**2.0)

INPUT_DIM = 28 * 28
OUTPUT_DIM = 26 #26  # num of classes

INPUT_WIDTH, HIDDEN_WIDTH = 1000, 400 #2000, 800 #1024, 128 #1000, 400 #500, 200 #1000, 400 #500, 200 #250, 100

lb, lw = 1e-2, 7.5 #1e-2,1,1 #1e-2, 1e-2, 1
meta = MetaData(input_dim = INPUT_DIM, input_width = INPUT_WIDTH, hidden_width = HIDDEN_WIDTH, output_dim = OUTPUT_DIM\
                , batch_size = BATCH_SIZE, lb = lb, lw = lw)

DEVICE = torch.device('cpu')  # change to `cuda:0` when available


#### L_b, L_w

In [ ]:
#for lb in , lw = 1e-2, 7.5
lbs = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1]
lws = [0.1, 0.5, 1, 2.5, 5, 7.5, 10, 20]
results = []
head = ["lb", "lw", "loss", "accuracy"]

for lb, lw in [(a,b) for a in lbs for b in lws]:
    meta = MetaData(input_dim = INPUT_DIM, input_width = INPUT_WIDTH, hidden_width = HIDDEN_WIDTH, output_dim = OUTPUT_DIM\
                    , batch_size = BATCH_SIZE, lb = lb, lw = lw)
    
    stepCalcZh = StepCalculatorZhangLambda(meta)
    stepCalcSoftmax = StepCalculatorEtaSoftmaxArmihoNorm2(meta)
    testNet = MNISTReLU(meta)
    testNet.set_log_level("info")
    testNet.set_slopes(slope_plus, slope_minus)
    testNet.init_weights(cb, cw)
    testNet.to(DEVICE)    
    # training loop
    for train_batch in train_dataloader:
        images, labels_raw = train_batch
        labels = torch.tensor([x-1 for x in labels_raw])
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        batch_size = images.shape[0]
        xx = images.view(batch_size, -1)

        with torch.no_grad():
            logits = testNet.forward_(xx)
            logits_detached = logits.detach().numpy()
            pp = labels_to_softhot(labels, meta.output_dim)
            qq = softmax(np.transpose(logits_detached), axis=(0))
            qq_active = reduce_to_active(qq, pp)

        qq_active_avg = np.average(qq_active)
        logging.info("##\n --==Initial qq for ones: min={}, max={}, avg={}==--"\
                        .format(np.min(qq_active), np.max(qq_active), qq_active_avg))
        if qq_active_avg > 0.90:
            logging.info("##Step with Zh-optim")
            eta, logits_latest = stepCalcZh.step(testNet, logits_detached, labels, xx)
        else:
            logging.info("##Step with norm2 optim")
            eta, logits_latest = stepCalcSoftmax.step(testNet, logits_detached, labels, xx)

    # testing loop
    test_loss_meter, test_accuracy_meter = AverageMeter(), AverageMeter(),
    for test_batch in test_dataloader:
        images, labels_raw = test_batch
        labels = torch.tensor([x-1 for x in labels_raw])
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        with torch.no_grad():
            batch_size = images.shape[0]
            xx = images.view(batch_size, -1)
            logits = testNet.forward_(xx)

            zz_logits = np.transpose(logits.detach().numpy())
            prediction = logits.argmax(dim=-1).detach()
            loss = loss_crossentropy(zz_logits, labels)
            test_loss_meter.update(loss)
            test_accuracy_meter.update(calculate_accuracy(prediction, labels))

    #results
    results.append([lb, lw, test_loss_meter.avg, test_accuracy_meter.avg])
    display.clear_output()
    print(tabulate(results, headers=head, tablefmt="grid"))


#### Eta_max

In [ ]:
#for lb in , lw = 1e-2, 7.5
eta_ratio_limits = [1.0, 1.25,1.5,1.75,2.0,2.25,2.5,2.75,3.0,3.5,4.0,5.0,0.0]
results = []
head = ["eta_max", "loss", "accuracy"]

eta = 0.0
for eta_ratio_limit in eta_ratio_limits:
    meta = MetaData(input_dim = INPUT_DIM, input_width = INPUT_WIDTH, hidden_width = HIDDEN_WIDTH, output_dim = OUTPUT_DIM\
                    , batch_size = BATCH_SIZE, lb = 0.005, lw = 5.0) #0.005, 5.0
    
    stepCalcZh = StepCalculatorZhangLambda(meta)
    stepCalcSoftmax = StepCalculatorEtaSoftmaxArmihoNorm2(meta)
    testNet = MNISTReLU(meta)
    testNet.set_log_level("info")
    testNet.set_slopes(slope_plus, slope_minus)
    testNet.init_weights(cb, cw)
    testNet.to(DEVICE)
    # training loop
    for train_batch in train_dataloader:
        images, labels_raw = train_batch
        labels = torch.tensor([x-1 for x in labels_raw])
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        batch_size = images.shape[0]
        xx = images.view(batch_size, -1)

        with torch.no_grad():
            logits = testNet.forward_(xx)
            logits_detached = logits.detach().numpy()
            pp = labels_to_softhot(labels, meta.output_dim)
            qq = softmax(np.transpose(logits_detached), axis=(0))
            qq_active = reduce_to_active(qq, pp)

        qq_active_avg = np.average(qq_active)
        logging.info("##\n --==Initial qq for ones: min={}, max={}, avg={}==--"\
                        .format(np.min(qq_active), np.max(qq_active), qq_active_avg))
        if qq_active_avg > 0.90:
            logging.info("##Step with Zh-optim")
            eta, logits_latest = stepCalcZh.step(testNet, logits_detached, labels, xx)
        else:
            logging.info("##Step with norm2 optim")
            eta, logits_latest = stepCalcSoftmax.step(testNet, logits_detached, labels, xx, eta*eta_ratio_limit)

    # testing loop
    test_loss_meter, test_accuracy_meter = AverageMeter(), AverageMeter(),
    for test_batch in test_dataloader:
        images, labels_raw = test_batch
        labels = torch.tensor([x-1 for x in labels_raw])
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        with torch.no_grad():
            batch_size = images.shape[0]
            xx = images.view(batch_size, -1)
            logits = testNet.forward_(xx)

            zz_logits = np.transpose(logits.detach().numpy())
            prediction = logits.argmax(dim=-1).detach()
            loss = loss_crossentropy(zz_logits, labels)
            test_loss_meter.update(loss)
            test_accuracy_meter.update(calculate_accuracy(prediction, labels))

    #results
    results.append([eta_ratio_limit, test_loss_meter.avg, test_accuracy_meter.avg])
    display.clear_output()
    print(tabulate(results, headers=head, tablefmt="grid"))


#### Debug for zero_grad

In [ ]:
def printData(direct, delta):
    print("Direct: min={}, max={}, abs aggr={}".format(np.min(direct), np.max(direct), np.sum(np.abs(direct))))
    print("Deltas: min={}, max={}, abs aggr={}".format(np.min(delta), np.max(delta), np.sum(np.abs(delta)))) 

testNetOpt = MNISTReLU(meta)
testNetOpt.set_log_level("info")
testNetOpt.set_slopes(slope_plus, slope_minus)
testNetOpt.init_weights(cb, cw)
testNetOpt.to(DEVICE)

testNetOpt.save_txt("tmp")
testNetMan = MNISTReLU(meta)
testNetMan.set_slopes(slope_plus, slope_minus)
testNetMan.init_weights_txt('tmp')
testNetMan.to(DEVICE)
#testNetMan.forward_(xx)

STEPS = 5

stepCalcZh = StepCalculatorZhangSimplified(meta)
#stepCalcSoftmax = StepCalculatorEtaSoftmaxArmihoNorm2(meta)
loss_fn = nn.CrossEntropyLoss(reduction='sum')
#loss_fnSoftmax = torch.linalg.matrix_norm(1-nn.LogSoftmax(dim=1), ord='fro')
optimizer = optim.SGD(testNetOpt.parameters(), lr=1e-2)

output0_fc_w = testNetOpt.output_fc.weight.detach().numpy()
output0_fc_b = testNetOpt.output_fc.bias.detach().numpy()
hidden0_fc_w = testNetOpt.hidden_fc.weight.detach().numpy()
hidden0_fc_b = testNetOpt.hidden_fc.bias.detach().numpy()
input0_fc_w = testNetOpt.input_fc.weight.detach().numpy()
input0_fc_b = testNetOpt.input_fc.bias.detach().numpy()

for step in range(STEPS):
    print("STEP NUMBER {}".format(step))
    images, labels_raw = next(iter(train_dataloader))
    labels = labels_raw - 1
    images = images.to(DEVICE)
    labels = labels.to(DEVICE)
    batch_size = images.shape[0]
    xx = images.view(batch_size, -1)

    logging.info("##Step with Zhang-torch optim")
    optimizer.zero_grad()
    logits1 = testNetOpt.forward_(xx)
    logits_detached = logits1.detach().numpy()
    loss = loss_fn(logits1, labels)
    loss.backward()
    output1OptDelta_fc_b = testNetOpt.output_fc.bias.grad.detach().numpy()
    print("Grad bias optim:\n{}".format(output1OptDelta_fc_b))
    etaOpt = stepCalcZh.calc_eta(testNetOpt, loss.item())

    optimizer.param_groups[0]['lr'] = etaOpt # set learning rate
    optimizer.step()
    output1Opt_fc_w = testNetOpt.output_fc.weight.detach().numpy()
    output1Opt_fc_b = testNetOpt.output_fc.bias.detach().numpy()
    hidden1Opt_fc_w = testNetOpt.hidden_fc.weight.detach().numpy()
    hidden1Opt_fc_b = testNetOpt.hidden_fc.bias.detach().numpy()
    input1Opt_fc_w = testNetOpt.input_fc.weight.detach().numpy()
    input1Opt_fc_b = testNetOpt.input_fc.bias.detach().numpy()

    logging.info("##Step with Zhang-direct optim")
    testNetMan.zero_grad()
    logits2 = testNetMan.forward_(xx)
    loss = stepCalcZh.criterion(logits2, labels)
    loss.backward()
    etaMan = stepCalcZh.calc_eta(testNetOpt, loss.item())
    stepCalcZh.calculate_deltasZhEta(testNetMan, etaMan)
    output1ManDelta_fc_b = stepCalcZh.delta_bias_02.numpy()/-etaMan
    print("Grad bias direct:\n{}".format(output1ManDelta_fc_b))
    stepCalcZh.do_step0(testNetMan, 1.0)
    #eta, logits_latest = stepCalcZh.step(testNetMan, logits_detached, labels, xx)
    output1Man_fc_w = testNetMan.output_fc.weight.detach().numpy() #.grad
    output1Man_fc_b = testNetMan.output_fc.bias.detach().numpy()
    hidden1Man_fc_w = testNetMan.hidden_fc.weight.detach().numpy()
    hidden1Man_fc_b = testNetMan.hidden_fc.bias.detach().numpy()
    input1Man_fc_w = testNetMan.input_fc.weight.detach().numpy()
    input1Man_fc_b = testNetMan.input_fc.bias.detach().numpy()


    print("Etas optim={}, man={}:".format(etaOpt, etaMan))
    print("Output weights:")
    printData(output1Man_fc_w, output1Opt_fc_w-output1Man_fc_w)
    print("Output biases:")
    printData(output1Man_fc_b, output1Opt_fc_b-output1Man_fc_b)

    print("Hidden weights:")
    printData(hidden1Man_fc_w, hidden1Opt_fc_w-hidden1Man_fc_w)
    print("Hidden biases:")
    printData(hidden1Man_fc_b, hidden1Opt_fc_b-hidden1Man_fc_b)

    print("Input weights:")
    printData(input1Man_fc_w, input1Opt_fc_w-input1Man_fc_w)
    print("Input biases:")
    printData(input1Man_fc_b, input1Opt_fc_b-input1Man_fc_b)